# Unbounded Graph Expansion — arunesh-mittal/VariationalRecurrentAutoEncoder

**Smell (`vrae.py` line 177):** The training loop calls `session.run([optimizer, loss], feed_dict=...)` in a plain Python `for` loop without resetting the TF graph between outer iterations. When the outer loop is the run-level loop (e.g. running multiple experiments), each outer call re-uses the same session with an ever-growing graph from intermediate ops, leading to unbounded memory growth.

**Fix:** Wrap each experiment run in `tf.compat.v1.reset_default_graph()` to tear down and rebuild the graph cleanly, preventing accumulation across runs. In TF2/Keras terms, use `K.clear_session()` + rebuild the model per run.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS     = 10
SEQ_LEN    = 20
INPUT_DIM  = 16
LATENT_DIM = 8
HIDDEN_DIM = 32
EPOCHS     = 3
BATCH      = 64
LR         = 1e-3

# Synthetic sequential data (proxy for VRAE time-series dataset)
np.random.seed(42)
N_SAMPLES = 3000
X_data = np.random.randn(N_SAMPLES, SEQ_LEN, INPUT_DIM).astype('float32')
split  = int(0.8 * N_SAMPLES)
X_tr, X_te = X_data[:split], X_data[split:]
print(f'Synthetic sequential data: train={X_tr.shape}  test={X_te.shape}')

In [ ]:
# VRAE: Variational Recurrent AutoEncoder (GRU-based, mirrors arunesh-mittal/vrae.py)
def build_vrae():
    # Encoder
    enc_inp = tf.keras.Input(shape=(SEQ_LEN, INPUT_DIM), name='enc_input')
    h       = tf.keras.layers.GRU(HIDDEN_DIM, name='enc_gru')(enc_inp)
    z_mean  = tf.keras.layers.Dense(LATENT_DIM, name='z_mean')(h)
    z_log_var = tf.keras.layers.Dense(LATENT_DIM, name='z_log_var')(h)

    def sampling(args):
        mu, lv = args
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * lv) * eps

    z = tf.keras.layers.Lambda(sampling, name='z')([z_mean, z_log_var])

    # Decoder
    z_rep   = tf.keras.layers.RepeatVector(SEQ_LEN, name='repeat')(z)
    dec_out = tf.keras.layers.GRU(HIDDEN_DIM, return_sequences=True, name='dec_gru')(z_rep)
    x_recon = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(INPUT_DIM), name='reconstruction')(dec_out)

    vrae = tf.keras.Model(enc_inp, [x_recon, z_mean, z_log_var])
    return vrae

def vrae_loss(x, x_recon, z_mean, z_log_var):
    recon = tf.reduce_mean(tf.square(x - x_recon))
    kl    = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
    return recon + kl

print('VRAE builder ready')

## BEFORE — Smell Active (model/optimizer reused across runs — graph accumulates ops)

In [ ]:
results_before = []

# ❌ SMELL (mirrors vrae.py line 177 pattern):
# Model and optimizer built ONCE outside the run loop.
# Across runs the graph accumulates gradient ops, variable refs, and
# intermediate tensors — memory grows unboundedly with each run.
smelly_model = build_vrae()
smelly_opt   = tf.keras.optimizers.Adam(LR)

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')

    tracker = EmissionsTracker(
        project_name=f'VRAE_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    n_batches = len(X_tr) // BATCH
    total_loss = 0.0
    for epoch in range(EPOCHS):
        idx = np.random.permutation(len(X_tr))
        ep_loss = 0.0
        for b in range(n_batches):
            xb = tf.constant(X_tr[idx[b*BATCH:(b+1)*BATCH]])
            with tf.GradientTape() as tape:
                x_recon, z_mean, z_log_var = smelly_model(xb, training=True)
                loss = vrae_loss(xb, x_recon, z_mean, z_log_var)
            grads = tape.gradient(loss, smelly_model.trainable_variables)
            smelly_opt.apply_gradients(zip(grads, smelly_model.trainable_variables))
            ep_loss += loss.numpy()
        ep_loss /= n_batches
        total_loss += ep_loss
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={ep_loss:.4f}')

    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'avg_loss': round(total_loss / EPOCHS, 6),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')

del smelly_model, smelly_opt; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/arunesh_VRAE_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/arunesh_VRAE_before.csv')

## AFTER — Smell Fixed (K.clear_session() + model rebuilt each run — no graph accumulation)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    # ✅ FIX: clear session + rebuild model each run — graph starts fresh,
    # no ops from prior runs accumulate
    K.clear_session(); gc.collect()
    model = build_vrae()
    opt   = tf.keras.optimizers.Adam(LR)

    tracker = EmissionsTracker(
        project_name=f'VRAE_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    n_batches = len(X_tr) // BATCH
    total_loss = 0.0
    for epoch in range(EPOCHS):
        idx = np.random.permutation(len(X_tr))
        ep_loss = 0.0
        for b in range(n_batches):
            xb = tf.constant(X_tr[idx[b*BATCH:(b+1)*BATCH]])
            with tf.GradientTape() as tape:
                x_recon, z_mean, z_log_var = model(xb, training=True)
                loss = vrae_loss(xb, x_recon, z_mean, z_log_var)
            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))
            ep_loss += loss.numpy()
        ep_loss /= n_batches
        total_loss += ep_loss
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={ep_loss:.4f}')

    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'avg_loss': round(total_loss / EPOCHS, 6),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model, opt; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/arunesh_VRAE_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/arunesh_VRAE_after.csv')

In [ ]:
print('\n=== SUMMARY: arunesh-mittal/VariationalRecurrentAutoEncoder ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")